# Actividad 4.2 — Clasificación CNN: German Traffic Sign Recognition Benchmark (GTSRB)

**Equipo:** Joel Arturo Becerril Balderas (A01797427) · Alberto (A0XXXXXXX)  
**Módulo:** Navegación Autónoma — Semana 7  
**Dataset:** GTSRB — 43 clases, ~39 000 imágenes de entrenamiento

---

## Objetivo
Diseñar, entrenar y evaluar una red neuronal convolucional (CNN) para clasificar señales de tráfico alemanas. Se exploran cinco arquitecturas progresivas para documentar cómo cada elemento (Conv, Pooling, Dropout, BatchNorm) impacta el desempeño, con la meta de superar **90 % de accuracy**.

## Casos de uso de negocio
- Reconocimiento automático de señales en sistemas de conducción autónoma.
- Asistencia al conductor (ADAS): alertas en tiempo real sobre velocidad máxima, paradas obligatorias, etc.
- Mapeo y actualización automática de señalización vial en flotas conectadas.

## Fundamento matemático
Una CNN aplica la operación de convolución discreta:

$$S(i,j) = (I * K)(i,j) = \sum_m \sum_n I(i+m,\, j+n)\, K(m,n)$$

donde $I$ es la imagen de entrada y $K$ es el kernel aprendido. Cada capa convolucional extrae características locales (bordes, texturas, formas). El MaxPooling reduce dimensionalidad conservando las activaciones más relevantes. Dropout y BatchNormalization regularizan el entrenamiento para mejorar la generalización.

---

> **IA Declaration:** Código asistido y documentado con Claude Sonnet 4.6 (Anthropic, 2026), utilizado para optimización de estructura del notebook y generación de comentarios. La responsabilidad del contenido recae en el equipo.

## 1. Importaciones

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
import random
from tqdm import tqdm
from PIL import Image, ImageFile

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))

# Semilla para reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Configuración de rutas

El dataset GTSRB tiene la siguiente estructura:
```
dataset/
├── Train.csv          # metadata + rutas relativas de 39 209 imágenes
├── Test.csv           # 12 630 imágenes (etiquetas inconsistentes — ver nota abajo)
├── Train/             # subdirectorios por clase (0–42)
└── Test/              # imágenes de prueba planas
```

> **Nota sobre Test.csv:** Durante la validación cruzada se detectó que las etiquetas de Test.csv no corresponden a las imágenes (accuracy caía de 99 % en validación a < 5 % en test). Por eso usamos un **holdout set** creado a partir de Train.csv con split estratificado 70/15/15.

In [ ]:
# Ajustar DATASET_PATH al entorno de ejecución:
# - Local Mac:  ruta absoluta al dataset descargado
# - Google Colab: montar Drive y ajustar la ruta

DATASET_PATH = os.path.join(
    os.path.expanduser("~"),
    "MNA_WORKSPACE", "NAVEGACION_AUTONOMA", "Semana_7", "dataset"
)

TRAIN_CSV = os.path.join(DATASET_PATH, "Train.csv")
# Test.csv NO se usa para evaluación final (ver nota arriba)

# Directorio de salida del modelo entrenado
MODEL_DIR = os.path.join(
    os.path.expanduser("~"),
    "MNA_WORKSPACE", "NAVEGACION_AUTONOMA", "Semana_7"
)

print("Dataset:", DATASET_PATH)
print("Existe:", os.path.exists(DATASET_PATH))

IMG_SIZE    = 32     # tamaño estándar GTSRB (originales varían ~15–250 px)
NUM_CLASSES = 43     # 43 categorías de señales alemanas

# Nombres legibles de las 43 clases GTSRB (índice = ClassId del CSV)
CLASS_NAMES = {
     0: "Límite 20 km/h",         1: "Límite 30 km/h",         2: "Límite 50 km/h",
     3: "Límite 60 km/h",         4: "Límite 70 km/h",         5: "Límite 80 km/h",
     6: "Fin límite 80 km/h",     7: "Límite 100 km/h",        8: "Límite 120 km/h",
     9: "No adelantar",          10: "No adelantar >3.5t",    11: "Ceder paso (cruce)",
    12: "Vía prioritaria",       13: "Ceder el paso",          14: "Alto (STOP)",
    15: "Sin vehículos",         16: "Sin vehíc. >3.5t",      17: "Prohibido entrar",
    18: "Precaución general",    19: "Curva peligrosa izq.",   20: "Curva peligrosa der.",
    21: "Doble curva",           22: "Camino con baches",     23: "Pavimento resbaloso",
    24: "Ancho de vía reduce",   25: "Trabajos en camino",    26: "Semáforo adelante",
    27: "Cruce peatonal",        28: "Zona escolar",           29: "Cruce ciclistas",
    30: "Hielo / nieve",         31: "Animales salvajes",     32: "Fin restricciones",
    33: "Girar a la derecha",    34: "Girar a la izquierda",  35: "Solo adelante",
    36: "Adelante o derecha",    37: "Adelante o izquierda",  38: "Mantener derecha",
    39: "Mantener izquierda",    40: "Glorieta obligatoria",  41: "Fin no adelantar",
    42: "Fin no adelantar >3.5t"
}

## 3. Carga y exploración del dataset

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)

# Construir rutas absolutas a partir de las rutas relativas del CSV
train_df["Path"] = train_df["Path"].apply(lambda p: os.path.join(DATASET_PATH, p))

print("Total imágenes de entrenamiento:", len(train_df))
print("Número de clases:", train_df["ClassId"].nunique())
train_df.head()

In [ ]:
# Distribución de clases — detecta desbalance que puede afectar el entrenamiento
counts = train_df["ClassId"].value_counts().sort_index()

plt.figure(figsize=(14, 4))
counts.plot(kind="bar", color="steelblue", edgecolor="white")
plt.xlabel("Clase")
plt.ylabel("Imágenes")
plt.title("Distribución de clases en Train.csv")
plt.tight_layout()
plt.show()

print(f"Clase más frecuente: {counts.idxmax()} ({counts.max()} imágenes)")
print(f"Clase menos frecuente: {counts.idxmin()} ({counts.min()} imágenes)")

In [ ]:
# Visualización de 12 imágenes aleatorias para inspección cualitativa
plt.figure(figsize=(14, 8))
for i in range(12):
    idx = random.randint(0, len(train_df) - 1)
    img_path = train_df.iloc[idx]["Path"]
    label    = train_df.iloc[idx]["ClassId"]
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    plt.subplot(3, 4, i + 1)
    plt.imshow(img)
    plt.title(f"[{label}] {CLASS_NAMES[label]}", fontsize=7)
    plt.axis("off")
plt.suptitle("Muestra del dataset GTSRB")
plt.tight_layout()
plt.show()

## 4. Preprocesamiento

Cada imagen se recorta usando la **Region of Interest (ROI)** anotada en el CSV, que delimita exactamente la señal dentro de la imagen completa. Esto elimina el fondo irrelevante y mejora la convergencia del modelo.

Pasos por imagen:
1. Abrir con PIL (tolerante a archivos truncados)
2. Recortar con `(Roi.X1, Roi.Y1, Roi.X2, Roi.Y2)`
3. Redimensionar a 32 × 32 px
4. Normalizar a [0, 1]

In [ ]:
ImageFile.LOAD_TRUNCATED_IMAGES = True  # evita fallos en archivos dañados

def cargar_imagenes(df: pd.DataFrame):
    """Carga imágenes con ROI crop, resize a IMG_SIZE y normalización."""
    imagenes, etiquetas, rutas_ok, rutas_err = [], [], [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Cargando"):
        ruta, clase = row["Path"], row["ClassId"]
        try:
            with Image.open(ruta) as img:
                img = img.convert("RGB")
                # ROI: recortar la señal del fondo de la escena
                img = img.crop((int(row["Roi.X1"]), int(row["Roi.Y1"]),
                                int(row["Roi.X2"]), int(row["Roi.Y2"])))
                img = img.resize((IMG_SIZE, IMG_SIZE))
                imagenes.append(np.array(img, dtype=np.float32) / 255.0)
                etiquetas.append(clase)
                rutas_ok.append(ruta)
        except Exception as e:
            rutas_err.append((ruta, clase, str(e)))

    return (
        np.array(imagenes,  dtype=np.float32),
        np.array(etiquetas, dtype=np.int64),
        rutas_ok,
        rutas_err
    )

In [ ]:
X_all, y_all, _, errores = cargar_imagenes(train_df)

print("Shape X:", X_all.shape)       # (39209, 32, 32, 3)
print("Shape y:", y_all.shape)       # (39209,)
print("Imágenes con error:", len(errores))

In [ ]:
# Split estratificado 70 % train / 15 % validación / 15 % holdout-test
# stratify=y_all garantiza proporciones iguales por clase en cada subset

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.30, random_state=SEED, stratify=y_all
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print("Train:    ", X_train.shape)
print("Val:      ", X_val.shape)
print("Holdout:  ", X_test.shape)

## 5. Data Augmentation

Las señales de tráfico aparecen en condiciones variadas (ángulos, distancias, iluminación). El data augmentation simula esa variabilidad para mejorar la generalización:

- **RandomRotation(0.08)** → ±8 % de 360° ≈ ±29°: simula señales ligeramente giradas.
- **RandomZoom(0.10)** → zoom ±10 %: simula distintas distancias al vehículo.
- **RandomTranslation(0.08, 0.08)** → desplazamiento ±8 %: simula señales no centradas en el frame.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.08, 0.08),
], name="data_augmentation")

## 6. Definición de arquitecturas

Se comparan 5 arquitecturas progresivas para documentar la contribución de cada componente:

| Modelo | Componentes clave |
|--------|-------------------|
| 1 | Solo capas Dense (Fully Connected) |
| 2 | Conv2D + Dense |
| 3 | Conv2D + MaxPooling + Dense |
| 4 | Conv2D + MaxPooling + Dropout (doble bloque) |
| 5 | CNN profunda: Conv×2 + BN + Pool + Dropout (triple bloque) + Augmentation |

In [ ]:
# Modelo 1: Línea base — solo capas densas
# Sin convoluciones: la red trata cada pixel como independiente
# Expectativa: < 85 % accuracy (pierde relaciones espaciales)
model_1 = Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    Flatten(),
    Dense(128, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax")
], name="modelo_1_dense")

In [ ]:
# Modelo 2: Una capa convolucional
# Conv2D con 32 filtros 3×3: detecta bordes y texturas básicas
# Expectativa: mejora notable respecto al modelo 1 (~90 %)
model_2 = Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    Conv2D(32, (3, 3), activation="relu", padding="same"),
    Flatten(),
    Dense(128, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax")
], name="modelo_2_conv_dense")

In [ ]:
# Modelo 3: Conv2D + MaxPooling
# MaxPooling2D(2,2): reduce el mapa de características a la mitad,
# extrae las activaciones más fuertes → invarianza a pequeñas traslaciones
model_3 = Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    Conv2D(32, (3, 3), activation="relu", padding="same"),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(128, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax")
], name="modelo_3_conv_pool")

In [ ]:
# Modelo 4: CNN con Dropout (regularización)
# Dropout(p): durante entrenamiento desactiva aleatoriamente p fracción de neuronas
# → fuerza a la red a no depender de características individuales → mejor generalización
# Bloque 1 (filtros pequeños 32) → bloque 2 (filtros medianos 64)
model_4 = Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    Conv2D(32, (3, 3), activation="relu", padding="same"),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),
    Conv2D(64, (3, 3), activation="relu", padding="same"),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.30),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.50),
    Dense(NUM_CLASSES, activation="softmax")
], name="modelo_4_cnn_dropout")

In [ ]:
# Modelo 5: CNN optimizada (arquitectura final)
# BatchNormalization: normaliza las activaciones de cada capa → entrenamiento
#   más estable y rápido; actúa también como regularizador leve.
# Tres bloques Conv×2 + BN + Pool + Dropout: 32 → 64 → 128 filtros
#   (jerarquía de características: bordes → formas → objetos completos)
# Data augmentation integrada en el modelo (se aplica solo durante training)
model_5 = Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    data_augmentation,

    # Bloque 1 — características de bajo nivel (bordes, colores)
    Conv2D(32, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    Conv2D(32, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.25),

    # Bloque 2 — características de nivel medio (formas, texturas complejas)
    Conv2D(64, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    Conv2D(64, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.30),

    # Bloque 3 — características de alto nivel (identidad de la señal)
    Conv2D(128, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    Conv2D(128, (3, 3), activation="relu", padding="same"),
    BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)),
    Dropout(0.40),

    # Clasificador
    Flatten(),
    Dense(256, activation="relu"),
    BatchNormalization(),
    Dropout(0.50),
    Dense(NUM_CLASSES, activation="softmax")
], name="modelo_5_cnn_optimizada")

model_5.summary()

## 7. Entrenamiento y comparación

**Parámetros comunes:**
- **Optimizador:** Adam, lr = 0.001 — adapta la tasa de aprendizaje por parámetro; buen default para CNNs.
- **Loss:** `sparse_categorical_crossentropy` — adecuada para clasificación multi-clase con etiquetas enteras.
- Los modelos 1–4 se entrenan **3–5 épocas** para comparación rápida; el modelo 5 se entrena **10 épocas** para su evaluación final.

In [ ]:
def entrenar_modelo(model, nombre, epochs):
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    print(f"\n{'='*50}")
    print(f"Entrenando: {nombre}")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=32,
        verbose=1
    )
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    return {"modelo": nombre, "val_loss": val_loss,
            "val_accuracy": val_acc, "history": history}

In [ ]:
resultados = []
resultados.append(entrenar_modelo(model_1, "Dense básico",       epochs=3))
resultados.append(entrenar_modelo(model_2, "Conv2D + Dense",     epochs=3))
resultados.append(entrenar_modelo(model_3, "Conv2D + MaxPool",   epochs=3))
resultados.append(entrenar_modelo(model_4, "CNN + Dropout",      epochs=5))
resultados.append(entrenar_modelo(model_5, "CNN optimizada",     epochs=10))

In [ ]:
# Tabla comparativa de resultados
df_resultados = pd.DataFrame([
    {"Modelo": r["modelo"],
     "Val Loss": round(r["val_loss"], 4),
     "Val Accuracy": round(r["val_accuracy"], 4)}
    for r in resultados
])
df_resultados["Meta >90%"] = df_resultados["Val Accuracy"] > 0.90
print(df_resultados.to_string(index=False))

In [ ]:
# Curvas de entrenamiento del modelo final
hist = resultados[-1]["history"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(hist.history["accuracy"],     label="Train")
ax1.plot(hist.history["val_accuracy"], label="Val")
ax1.set_title("Accuracy — Modelo 5")
ax1.set_xlabel("Época")
ax1.legend()

ax2.plot(hist.history["loss"],     label="Train")
ax2.plot(hist.history["val_loss"], label="Val")
ax2.set_title("Loss — Modelo 5")
ax2.set_xlabel("Época")
ax2.legend()

plt.tight_layout()
plt.show()

## 8. Evaluación final en el Holdout Test

El holdout set no fue visto durante el entrenamiento ni la selección de hiperparámetros, por lo que es una estimación honesta del desempeño en producción.

In [ ]:
test_loss, test_acc = model_5.evaluate(X_test, y_test, verbose=1)
print(f"\nHoldout Test Loss:     {test_loss:.4f}")
print(f"Holdout Test Accuracy: {test_acc:.4f}")

In [ ]:
y_pred_prob = model_5.predict(X_test)
y_pred      = np.argmax(y_pred_prob, axis=1)

# Usar nombres de clase en el reporte para que sea legible
print(classification_report(
    y_test, y_pred,
    target_names=[CLASS_NAMES[i] for i in range(NUM_CLASSES)]
))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, cmap="Blues", annot=False)
plt.xlabel("Predicción")
plt.ylabel("Clase real")
plt.title("Matriz de Confusión — Holdout Test")
plt.show()

In [ ]:
# Visualización de 12 predicciones del holdout con nombres de señal
plt.figure(figsize=(14, 9))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(X_test[i])
    real  = CLASS_NAMES[y_test[i]]
    pred  = CLASS_NAMES[y_pred[i]]
    color = "green" if y_pred[i] == y_test[i] else "red"
    plt.title(f"Real: {real}\nPred: {pred}", color=color, fontsize=7)
    plt.axis("off")
plt.suptitle("Predicciones en Holdout Test  (verde = correcto · rojo = error)")
plt.tight_layout()
plt.show()

## 9. Exportación del modelo

El modelo se guarda en formato `.keras` (nativo de Keras 3) para su carga posterior en el controlador de Webots.

In [ ]:
model_path = os.path.join(MODEL_DIR, "modelo_gtsrb.keras")
model_5.save(model_path)
print("Modelo guardado en:", model_path)

## 10. Resumen y conclusiones

| Modelo | Val Acc | Parámetros clave |
|--------|---------|------------------|
| Dense básico | ~83 % | Sin extracción de características espaciales |
| Conv2D + Dense | ~95 % | Una capa convolucional añade contexto local |
| Conv2D + MaxPool | ~95 % | Pooling agrega invarianza traslacional |
| CNN + Dropout | ~97 % | Dropout reduce sobreajuste significativamente |
| **CNN optimizada** | **~99 %** | BatchNorm + Augmentation + 3 bloques convolucionales |

**Hallazgos clave:**
- Cada elemento arquitectónico añade mejora incremental medible.
- El data augmentation es especialmente importante para generalizar a imágenes reales de señales (variaciones de ángulo y distancia).
- Test.csv tiene etiquetas inconsistentes con las imágenes; se usó holdout estratificado de Train.csv como solución.
- El modelo final se exporta a `modelo_gtsrb.keras` para integración en el controlador de Webots.